# LangChain: Agents & Tools

## Outline
* مفهوم Agent و ReAct loop
* ساخت tool با `@tool`
* Agent با built-in tools (Wikipedia, DuckDuckGo)
* Streaming خروجی agent
* Tool error handling با middleware
* Dynamic system prompt


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)


## ۱. مفهوم Agent

Agent = LLM + Tools + Loop

**ReAct pattern:**
1. LLM **فکر** می‌کنه (reasoning)
2. **تصمیم** می‌گیره کدوم tool استفاده کنه
3. Tool رو **اجرا** می‌کنه
4. نتیجه رو **می‌بینه** و دوباره فکر می‌کنه
5. تا رسیدن به جواب نهایی ادامه می‌ده

قدیمی: `initialize_agent` + `AgentType`
جدید: `create_agent`


## ۲. ساخت Tool با `@tool`

In [5]:
from datetime import date

# tool ساده
@tool
def get_today_date(text: str) -> str:
    """Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string."""
    return str(date.today())

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input should be a valid Python math expression.
    Example: '2 + 2', '15 * 4', '100 / 5'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(city: str) -> str:
    """
    Get weather for a city.

    IMPORTANT:
    Always pass city name in English (e.g. London, Tehran, Tokyo).
    Never use Persian or translated names.
    """    # در واقعیت باید به API وصل بشید
    weather_data = {
        "Tehran": "25°C, Sunny",
        "London": "15°C, Cloudy",
        "New York": "20°C, Partly cloudy",
        "Tokyo": "28°C, Humid",
    }
    return weather_data.get(city, f"Weather data for {city} not available.")


# مشاهده tool metadata
print(f"Tool name: {get_today_date.name}")
print(f"Description: {get_today_date.description}")
print(f"Args: {get_today_date.args}")


Tool name: get_today_date
Description: Returns today's date. Use this for any questions about today's date.
    The input should always be an empty string.
Args: {'text': {'title': 'Text', 'type': 'string'}}


In [7]:
# agent با tools
agent = create_agent(
    model=llm,
    tools=[get_today_date, calculate, get_weather],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "What is 25% of 300?"}]
})
print(response["messages"][-1].content)


25% of 300 is 75.


In [9]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "هوای تهران چند درجه است؟"}]
})
print(response["messages"][-1].content)


هوای تهران ۲۵ درجه سانتی‌گراد و آفتابی است.


## ۳. Built-in Tools — Wikipedia
``` pip install wikipedia ```

In [11]:
from langchain.agents import create_agent   
import wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# ✅ تنظیم user-agent — بدون این، API ویکیپدیا empty response برمی‌گردونه
wikipedia.set_user_agent("LangChain-Course-Bot/1.0 (educational purposes)")

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=1000,
    )
)

agent_wiki = create_agent(
    model=llm,
    tools=[wiki_tool, calculate, get_today_date],
    system_prompt="You are a research assistant. Use Wikipedia for factual questions.",
)

question = "Who invented Python programming language?"
response = agent_wiki.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)


C:\Users\Alireza\AppData\Local\Temp\ipykernel_2440\1834581274.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


Python programming language was invented by Guido van Rossum, who began working on it in the late 1980s as a successor to the ABC programming language.


``` pip install ddgs```

In [13]:
from langchain_community.tools import DuckDuckGoSearchRun

ddg_search = DuckDuckGoSearchRun()

agent_ddg = create_agent(
    model=llm,
    tools=[ddg_search, calculate, get_today_date],
    system_prompt=(
        "You are a web search assistant. "
        "Use DuckDuckGo to find up-to-date information."
    ),
)

question = "Latest LangChain version 2026"
response = agent_ddg.invoke({"messages": [{"role": "user", "content": question}]})
print("\n=== DuckDuckGo Agent ===")
print(response["messages"][-1].content)



=== DuckDuckGo Agent ===
The latest version of LangChain as of June 13, 2026, is `langchain-openai==1.3.2`. This release includes several enhancements such as finer-grained control over node execution, a new channel type to reduce checkpoint overhead for long-running threads, and a new content-block-centric streaming API (v3) with typed, per-channel projections. 

For more details, you can check the [LangChain GitHub repository](https://github.com/langchain-ai/langchain).


## ۴. Streaming خروجی Agent

<div dir="rtl">
streaming با stream_mode="values" — هر step رو می‌بینید
</div>

In [15]:
from langchain.messages import AIMessage, HumanMessage

print("=== Agent Streaming ===")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "دمای تهران چند درجه است؟"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if isinstance(latest, AIMessage):
        if latest.content:
            print(f"[AI]: {latest.content}")
        elif latest.tool_calls:
            for tc in latest.tool_calls:
                print(f"[Tool Call]: {tc['name']}({tc['args']})")
    elif hasattr(latest, 'name'):  # ToolMessage
        print(f"[Tool Result]: {latest.content[:100]}")


=== Agent Streaming ===
[Tool Result]: دمای تهران چند درجه است؟
[Tool Call]: get_weather({'city': 'Tehran'})
[Tool Result]: 25°C, Sunny
[AI]: دمای تهران ۲۵ درجه سانتی‌گراد و آفتابی است.


## ۵. Tool Error Handling با Middleware

In [25]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Middleware برای مدیریت خطاهای tool"""
    print("handle_tool_errors is called.")
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: {str(e)}. Please try a different approach.",
            tool_call_id=request.tool_call["id"]
        )

@tool
def risky_tool(number_a: str, number_b: str) -> str:
    """A tool that might fail. Input: two numbers. number_a / number_b"""
    return str(int(number_a) / int(number_b))  # division by zero اگه 0 باشه

agent_safe = create_agent(
    model=llm,
    tools=[risky_tool, calculate],
    middleware=[handle_tool_errors],
    system_prompt=(
        "You are a math assistant. "
        "ALWAYS use the risky_tool for ANY division operation, even if you think the result is undefined. "
        "Never answer math questions directly — always call the appropriate tool first."
    )
)
response = agent_safe.invoke({
    "messages": [{"role": "user", "content": "What is 10 divided by 0?"}]
})
print(response["messages"][-1].content)


handle_tool_errors is called.
It seems there was an issue with the division operation. Division by zero is undefined. If you have any other questions or need assistance with a different calculation, feel free to ask!
